In [3]:
# 1. Import libraries
import pandas as pd
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# import fasttext

In [4]:
# 2. Load and merge datasets
print('Loading training datasets...')
train_texts = pd.read_csv('training_datasets/train_text.csv')
train_labels = pd.read_csv('training_datasets/train_labels.csv')
train_df = pd.merge(train_texts, train_labels, on='id')

print('Loading testing datasets...')
test_texts = pd.read_csv('testing_datasets/test_text.csv')
test_labels = pd.read_csv('testing_datasets/test_labels.csv')
test_df = pd.merge(test_texts, test_labels, on='id')

print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

# Ensure text column is treated as string and handle nulls if any
train_df['text'] = train_df['text'].astype(str).fillna('')
test_df['text'] = test_df['text'].astype(str).fillna('')

Loading training datasets...
Loading testing datasets...
Train size: 146069, Test size: 36518


In [5]:
# 3. TF-IDF Vectorization
# max_df=0.85 removes words appearing in >85% of documents (like generic stopwords)
# min_df=5 removes words appearing <5 times overall (like typos)
print('Vectorizing text with TF-IDF...')
tfidf = TfidfVectorizer(max_df=0.85, min_df=5, ngram_range=(1, 2))

start_time = time.time()
X_train = tfidf.fit_transform(train_df['text'])
tfidf_time = time.time() - start_time
print(f'TF-IDF fit_transform time: {tfidf_time:.2f} seconds')

X_test = tfidf.transform(test_df['text'])

y_train = train_df['label']
y_test = test_df['label']

print(f'Vocabulary size: {len(tfidf.vocabulary_)}')


Vectorizing text with TF-IDF...
TF-IDF fit_transform time: 6.46 seconds
Vocabulary size: 141850


In [6]:
# 4. Train and Evaluate Multinomial Naive Bayes
print('Training Multinomial Naive Bayes...')
mnb_model = MultinomialNB()

start_time = time.time()
mnb_model.fit(X_train, y_train)
mnb_train_time = time.time() - start_time

start_time = time.time()
mnb_preds = mnb_model.predict(X_test)
mnb_test_time = time.time() - start_time

# Use macro average to treat all classes equally (good for slightly unbalanced datasets)
mnb_acc = accuracy_score(y_test, mnb_preds)
mnb_prec = precision_score(y_test, mnb_preds, average='macro')
mnb_rec = recall_score(y_test, mnb_preds, average='macro')
mnb_f1 = f1_score(y_test, mnb_preds, average='macro')

print(f'MNB Accuracy: {mnb_acc:.4f}')


Training Multinomial Naive Bayes...
MNB Accuracy: 0.8491


In [7]:
# 5. Train and Evaluate Logistic Regression
print('Training Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000)

start_time = time.time()
lr_model.fit(X_train, y_train)
lr_train_time = time.time() - start_time

start_time = time.time()
lr_preds = lr_model.predict(X_test)
lr_test_time = time.time() - start_time

lr_acc = accuracy_score(y_test, lr_preds)
lr_prec = precision_score(y_test, lr_preds, average='macro')
lr_rec = recall_score(y_test, lr_preds, average='macro')
lr_f1 = f1_score(y_test, lr_preds, average='macro')

print(f'LR Accuracy: {lr_acc:.4f}')


Training Logistic Regression...
LR Accuracy: 0.8892


In [ ]:
# 6. FastText Implementation (Placeholder for now)
# import fasttext
#
# print('Training FastText...')
# start_time = time.time()
# ft_model = fasttext.train_supervised(input='training_datasets/train_fasttext.txt', epoch=25, lr=0.1, wordNgrams=2, verbose=2)
# ft_train_time = time.time() - start_time
#
# print('Testing FastText...')
# start_time = time.time()
# ft_results = ft_model.test('testing_datasets/test_fasttext.txt')
# ft_test_time = time.time() - start_time
#
# ft_samples, ft_prec, ft_rec = ft_results
# ft_f1 = 2 * (ft_prec * ft_rec) / (ft_prec + ft_rec) if (ft_prec + ft_rec) > 0 else 0
# print(f'FastText Precision: {ft_prec:.4f}')


In [8]:
# 7. Compare Models
results_data = {
    'Model': ['Multinomial Naive Bayes', 'Logistic Regression'], # Add 'FastText' later
    'Accuracy': [mnb_acc, lr_acc],
    'Precision (Macro)': [mnb_prec, lr_prec],
    'Recall (Macro)': [mnb_rec, lr_rec],
    'F1-Score (Macro)': [mnb_f1, lr_f1],
    'Train Time (s)': [mnb_train_time, lr_train_time],
    'Test Time (s)': [mnb_test_time, lr_test_time]
}

results_df = pd.DataFrame(results_data)
results_df

,Model,Accuracy,Precision (Macro),Recall (Macro),F1-Score (Macro),Train Time (s),Test Time (s)
0,Multinomial Naive Bayes,0.849088,0.840402,0.811108,0.822723,0.027230,0.006319
1,Logistic Regression,0.889178,0.894380,0.852507,0.868608,4.014388,0.002039
